# CADEC → RQ instance adapter

Convert CADEC v2 (`text/` + `sct/`) into an instances CSV aligned to the **common**
pipeline schema shared by BioASQ and MedMentions.

- Core (BioASQ ∩ MedMentions): `instance_id`, `mention_context`, `gold_mention`, `gold_cui`
- MedMentions shared extras (when applicable): `document_id`, `full_original_text`,
  `semantic_type`, `mention_start`, `mention_end`, `original_text`
- **Not** included: BioASQ QA-only `question` / `gold_answer`

Does **not** generate perturbations.

In [1]:
# === Setup — absolute project root (nbconvert-safe) ===
import json
import os
import pickle
import re
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.home() / "projects" / "Measuring-Semantic-Stability-in-Clinical-LLMs"
CONFIG_PATH = PROJECT_ROOT / "config.json"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CONFIG_PATH:  {CONFIG_PATH}")
assert CONFIG_PATH.is_file(), f"Missing config.json: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)


def _expand_tree(obj):
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()


CADEC_ROOT = Path.home() / "data" / "cadec" / "data" / "cadec"
TEXT_DIR = CADEC_ROOT / "text"
SCT_DIR = CADEC_ROOT / "sct"
CACHE_DIR = Path.home() / "data" / "cadec"
SCTID_CUI_CACHE = CACHE_DIR / "sctid_to_cui.pkl"
CUI_STY_CACHE = CACHE_DIR / "cui_to_semantic_type.pkl"

UMLS_META = Path(
    os.environ.get("UMLS_META")
    or CFG.get("umls_meta")
    or str(Path.home() / "data" / "umls" / "2026AA" / "2026AA" / "META")
)
UMLS_META = Path(os.path.expanduser(str(UMLS_META)))
MRCONSO = UMLS_META / "MRCONSO.RRF"
MRSTY = UMLS_META / "MRSTY.RRF"

OUT_DIR = PROJECT_ROOT / "outputs" / "rq3" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "rq3_cadec_instances.csv"
BIOASQ_REF = OUT_DIR / "rq3_bioasq_instances.csv"
MM_REF = PROJECT_ROOT / "outputs" / "rq1" / "intermediate" / "rq1_sampled_instances.csv"

for label, p in [
    ("CADEC text/", TEXT_DIR),
    ("CADEC sct/", SCT_DIR),
    ("MRCONSO.RRF", MRCONSO),
    ("MRSTY.RRF", MRSTY),
    ("BioASQ ref", BIOASQ_REF),
    ("MedMentions ref", MM_REF),
]:
    print(f"Path [{label}]: {p}")
    assert p.exists(), f"Missing required path [{label}]: {p}"

# ---- Print both headers and align to the common / shared set ----
BIO_COLS = list(pd.read_csv(BIOASQ_REF, nrows=0).columns)
MM_COLS = list(pd.read_csv(MM_REF, nrows=0).columns)
print("\nBioASQ headers:")
print(" ", BIO_COLS)
print("MedMentions headers:")
print(" ", MM_COLS)

CORE_COLS = ["instance_id", "mention_context", "gold_mention", "gold_cui"]
common = [c for c in BIO_COLS if c in MM_COLS]
print("\nIntersection (BioASQ ∩ MedMentions):", common)
assert all(c in common for c in CORE_COLS), (
    f"Core columns {CORE_COLS} missing from intersection {common}"
)

# QA-only BioASQ columns — explicitly excluded for CADEC
QA_ONLY = ["question", "gold_answer"]
print("Excluded QA-only BioASQ columns:", QA_ONLY)

# MedMentions columns that also apply to CADEC NER-style instances
MM_SHARED = [
    c
    for c in [
        "document_id",
        "full_original_text",
        "semantic_type",
        "mention_start",
        "mention_end",
        "original_text",
    ]
    if c in MM_COLS
]
print("MedMentions shared extras to include:", MM_SHARED)

# Final output column order: MM-style layout for shared fields, core required
OUT_COLS = []
for c in [
    "instance_id",
    "document_id",
    "full_original_text",
    "mention_context",
    "gold_mention",
    "gold_cui",
    "semantic_type",
    "mention_start",
    "mention_end",
    "original_text",
]:
    if c in CORE_COLS or c in MM_SHARED:
        OUT_COLS.append(c)
print("\nCADEC output columns:", OUT_COLS)
assert all(c in OUT_COLS for c in CORE_COLS)
_log("Setup OK.")

PROJECT_ROOT: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs
CONFIG_PATH:  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json
Path [CADEC text/]: /home/s224858267/data/cadec/data/cadec/text
Path [CADEC sct/]: /home/s224858267/data/cadec/data/cadec/sct
Path [MRCONSO.RRF]: /home/s224858267/data/umls/2026AA/2026AA/META/MRCONSO.RRF
Path [MRSTY.RRF]: /home/s224858267/data/umls/2026AA/2026AA/META/MRSTY.RRF
Path [BioASQ ref]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_bioasq_instances.csv
Path [MedMentions ref]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/intermediate/rq1_sampled_instances.csv

BioASQ headers:
  ['question', 'gold_answer', 'instance_id', 'mention_context', 'gold_mention', 'gold_cui']
MedMentions headers:
  ['instance_id', 'document_id', 'pmid', 'title', 'abstract', 'full_original_text', 'mention_context', 'gold_mention', '

## 1) Parse `sct/*.ann` mentions

Format: `TT1<TAB><SCTID> | <pref_name> | <start> <end><TAB><mention_text>`  
Also handles `CONCEPT_LESS`, multi-code lines (`| + |` / `| or |`), discontinuous spans (`a b;c d`), and rare space-delimited rows.

In [2]:
# === Parse CADEC SNOMED annotations ==========================================
_OFFSET_RE = re.compile(r"(\d+\s+\d+(?:\s*;\s*\d+\s+\d+)*)\s*$")
_CODE_RE = re.compile(r"(\d+)\s*\|\s*([^|]+?)\s*\|")


def parse_ann_line(line: str, doc_id: str):
    """Parse one .ann line -> (record_dict | None, status)."""
    raw = line.rstrip("\n\r")
    if not raw.strip():
        return None, "empty"

    if "\t" in raw:
        parts = raw.split("\t")
    else:
        parts = re.split(r"\s{2,}", raw.strip())

    if len(parts) < 2:
        return None, "malformed"

    ann_id = parts[0].strip()
    if len(parts) >= 3:
        body = parts[1].strip()
        mention_text = parts[-1].strip()
    else:
        body = parts[1].strip()
        mention_text = ""

    if body.startswith("CONCEPT_LESS") or body.split(None, 1)[0] == "CONCEPT_LESS":
        return None, "concept_less"

    m_off = _OFFSET_RE.search(body)
    if not m_off:
        return None, "no_offsets"

    offsets_str = m_off.group(1)
    code_part = body[: m_off.start()].strip()
    if not code_part.endswith("|"):
        code_part = code_part + "|"

    valid = []
    for m in _CODE_RE.finditer(code_part):
        sctid, pref = m.group(1), m.group(2).strip()
        pref = re.sub(r"\s+", " ", pref).strip(" +")
        if not sctid.isdigit():
            continue
        if not pref or pref.lower() in {"or", "and"}:
            continue
        valid.append((sctid, pref))

    if not valid:
        if "CONCEPT_LESS" in body:
            return None, "concept_less"
        return None, "no_valid_sctid"

    sctid, pref_name = valid[0]  # first valid SCTID on multi-code lines
    spans = [(int(a), int(b)) for a, b in re.findall(r"(\d+)\s+(\d+)", offsets_str)]
    if not spans:
        return None, "no_offsets"

    char_start, char_end = spans[0]
    return {
        "doc_id": doc_id,
        "annotation_id": ann_id,
        "sctid": sctid,
        "snomed_pref_name": pref_name,
        "char_start": char_start,
        "char_end": char_end,
        "span_start": min(s for s, _ in spans),
        "span_end": max(e for _, e in spans),
        "mention_text": mention_text,
        "n_sctids_on_line": len(valid),
    }, "kept"


drop_counts = Counter()
parsed_mentions = []
ann_files = sorted(SCT_DIR.glob("*.ann"))
_log(f"Parsing {len(ann_files):,} annotation files from {SCT_DIR}")

for ann_path in ann_files:
    doc_id = ann_path.stem
    for line in ann_path.read_text(encoding="utf-8", errors="replace").splitlines():
        rec, status = parse_ann_line(line, doc_id)
        drop_counts[status] += 1
        if status == "kept":
            parsed_mentions.append(rec)

n_kept = drop_counts["kept"]
n_dropped = sum(v for k, v in drop_counts.items() if k != "kept")
_log(f"Mentions kept={n_kept:,} | dropped={n_dropped:,}")
print("Drop / keep breakdown:")
for k, v in sorted(drop_counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {k:20s} {v:,}")
sys.stdout.flush()
assert n_kept > 0, "No mentions kept after parsing — check CADEC sct/ format."
df_mentions = pd.DataFrame(parsed_mentions)
print(df_mentions.head(3).to_string())
sys.stdout.flush()

[2026-07-28 02:19:39 UTC] Parsing 1,250 annotation files from /home/s224858267/data/cadec/data/cadec/sct


[2026-07-28 02:19:41 UTC] Mentions kept=8,665 | dropped=446


Drop / keep breakdown:
  kept                 8,665
  concept_less         445
  no_valid_sctid       1


        doc_id annotation_id      sctid                      snomed_pref_name  char_start  char_end  span_start  span_end           mention_text  n_sctids_on_line
0  ARTHROTEC.1           TT1  271782001                                Drowsy           9        19           9        19             bit drowsy                 1
1  ARTHROTEC.1           TT2  246636008                 Blurred vision - hazy          29        50          29        50  little blurred vision                 1
2  ARTHROTEC.1           TT4  162076009  Excessive upper gastrointestinal gas          62        78          62        78       gastric problems                 1


## 2) SNOMED → CUI (MRCONSO) and CUI → semantic_type (MRSTY)

In [3]:
# === SCTID -> CUI (SNOMEDCT_US) and CUI -> TUI ===============================
def build_sctid_to_cui(mrconso_path: Path) -> dict:
    mapping = {}
    n_rows = 0
    with open(mrconso_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            if len(parts) < 15 or parts[11] != "SNOMEDCT_US":
                continue
            n_rows += 1
            code, cui = parts[13], parts[0]
            if code and cui and code not in mapping:
                mapping[code] = cui
            if n_rows % 500_000 == 0:
                _log(f"  MRCONSO SNOMEDCT_US rows={n_rows:,} | unique SCTIDs={len(mapping):,}")
    return mapping


def build_cui_to_sty(mrsty_path: Path, needed_cuis: set) -> dict:
    """First TUI per CUI (MedMentions-style semantic_type, e.g. T017)."""
    mapping = {}
    with open(mrsty_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            if len(parts) < 2:
                continue
            cui, tui = parts[0], parts[1]
            if cui in needed_cuis and cui not in mapping:
                mapping[cui] = tui
                if len(mapping) >= len(needed_cuis):
                    break
    return mapping


if SCTID_CUI_CACHE.is_file():
    _log(f"Loading cached SCTID→CUI map: {SCTID_CUI_CACHE}")
    with open(SCTID_CUI_CACHE, "rb") as _f:
        sctid_to_cui = pickle.load(_f)
    assert isinstance(sctid_to_cui, dict)
    _log(f"Cache loaded: {len(sctid_to_cui):,} SCTIDs")
else:
    _log(f"Building SCTID→CUI from {MRCONSO} (SAB==SNOMEDCT_US) …")
    sctid_to_cui = build_sctid_to_cui(MRCONSO)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with open(SCTID_CUI_CACHE, "wb") as _f:
        pickle.dump(sctid_to_cui, _f, protocol=pickle.HIGHEST_PROTOCOL)
    _log(f"Wrote {SCTID_CUI_CACHE} | {len(sctid_to_cui):,} SCTIDs")

df_mentions["cui"] = df_mentions["sctid"].map(sctid_to_cui)
mapped = df_mentions["cui"].notna()
n_map_ok = int(mapped.sum())
n_map_fail = int((~mapped).sum())
n_unique_sct_fail = int(df_mentions.loc[~mapped, "sctid"].nunique())
_log(
    f"CUI map: ok={n_map_ok:,} fail={n_map_fail:,} "
    f"(unique unmapped SCTIDs={n_unique_sct_fail:,}; often retired / non-US codes)"
)
if n_map_fail:
    print("Top unmapped SCTIDs:")
    print(
        df_mentions.loc[~mapped, ["sctid", "snomed_pref_name"]]
        .value_counts()
        .head(10)
        .to_string()
    )
    sys.stdout.flush()
assert n_map_ok > 0, "Zero SCTIDs mapped to CUIs — check MRCONSO / SAB filter."

needed_cuis = set(df_mentions.loc[mapped, "cui"].astype(str))
if CUI_STY_CACHE.is_file():
    _log(f"Loading cached CUI→semantic_type: {CUI_STY_CACHE}")
    with open(CUI_STY_CACHE, "rb") as _f:
        cui_to_sty = pickle.load(_f)
    # refresh any newly needed CUIs missing from cache
    missing = needed_cuis - set(cui_to_sty)
    if missing:
        _log(f"Refreshing STY for {len(missing):,} CUIs not in cache …")
        cui_to_sty.update(build_cui_to_sty(MRSTY, missing))
        with open(CUI_STY_CACHE, "wb") as _f:
            pickle.dump(cui_to_sty, _f, protocol=pickle.HIGHEST_PROTOCOL)
else:
    _log(f"Building CUI→semantic_type from {MRSTY} for {len(needed_cuis):,} CUIs …")
    cui_to_sty = build_cui_to_sty(MRSTY, needed_cuis)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with open(CUI_STY_CACHE, "wb") as _f:
        pickle.dump(cui_to_sty, _f, protocol=pickle.HIGHEST_PROTOCOL)
    _log(f"Wrote {CUI_STY_CACHE} | {len(cui_to_sty):,} CUIs")

df_mentions["semantic_type"] = df_mentions["cui"].map(cui_to_sty)
n_sty = int(df_mentions["semantic_type"].notna().sum())
_log(f"semantic_type filled for {n_sty:,} / {n_map_ok:,} mapped mentions")

[2026-07-28 02:19:41 UTC] Building SCTID→CUI from /home/s224858267/data/umls/2026AA/2026AA/META/MRCONSO.RRF (SAB==SNOMEDCT_US) …


[2026-07-28 02:19:46 UTC]   MRCONSO SNOMEDCT_US rows=500,000 | unique SCTIDs=146,546


[2026-07-28 02:19:50 UTC]   MRCONSO SNOMEDCT_US rows=1,000,000 | unique SCTIDs=306,253


[2026-07-28 02:19:58 UTC]   MRCONSO SNOMEDCT_US rows=1,500,000 | unique SCTIDs=457,047


[2026-07-28 02:20:01 UTC] Wrote /home/s224858267/data/cadec/sctid_to_cui.pkl | 537,781 SCTIDs


[2026-07-28 02:20:01 UTC] CUI map: ok=7,006 fail=1,659 (unique unmapped SCTIDs=125; often retired / non-US codes)


Top unmapped SCTIDs:
sctid              snomed_pref_name
3877011000036101   Lipitor             1073
3384011000036100   Arthrotec             62
3904011000036106   Zocor                 48
77424011000036100  ubidecarenone         42
3848011000036104   Pravachol             31
28551000168108     Voltaren              27
77435011000036104  ascorbic acid         22
4031011000036106   Crestor               20
21930011000036101  ezetimibe             17
21417011000036105  fenofibrate           15


[2026-07-28 02:20:01 UTC] Building CUI→semantic_type from /home/s224858267/data/umls/2026AA/2026AA/META/MRSTY.RRF for 881 CUIs …


[2026-07-28 02:20:03 UTC] Wrote /home/s224858267/data/cadec/cui_to_semantic_type.pkl | 881 CUIs


[2026-07-28 02:20:03 UTC] semantic_type filled for 7,006 / 7,006 mapped mentions


## 3) Build contexts and write instances CSV

In [ ]:
# === Sentence context + instances CSV ========================================
def surrounding_sentence(text: str, start: int, end: int) -> str:
    """Return the sentence / newline block covering [start, end)."""
    if not text:
        return ""
    n = len(text)
    start = max(0, min(int(start), n))
    end = max(start, min(int(end), n))

    left_nl = text.rfind("\n", 0, start)
    right_nl = text.find("\n", end)
    block_l = left_nl + 1 if left_nl >= 0 else 0
    block_r = right_nl if right_nl >= 0 else n
    block = text[block_l:block_r]
    rel_start = start - block_l
    rel_end = end - block_l

    terminators = ".!?"
    s = rel_start
    while s > 0 and block[s - 1] not in terminators:
        s -= 1
    e = rel_end
    while e < len(block) and (e == 0 or block[e - 1] not in terminators):
        e += 1
    sent = block[s:e].strip()
    if not sent:
        pad = 80
        sent = text[max(0, start - pad) : min(n, end + pad)].strip()
    return re.sub(r"\s+", " ", sent)


text_by_doc = {}
missing_text = 0
for doc_id in df_mentions["doc_id"].unique():
    tp = TEXT_DIR / f"{doc_id}.txt"
    if not tp.is_file():
        missing_text += 1
        text_by_doc[doc_id] = ""
    else:
        text_by_doc[doc_id] = tp.read_text(encoding="utf-8", errors="replace")
_log(f"Loaded texts for {len(text_by_doc):,} docs (missing .txt={missing_text})")

df_ok = df_mentions[df_mentions["cui"].notna()].copy()
rows = []
for r in df_ok.itertuples(index=False):
    full = text_by_doc.get(r.doc_id, "")
    ctx = surrounding_sentence(full, r.span_start, r.span_end)
    gold_mention = r.mention_text if r.mention_text else full[r.char_start : r.char_end]
    cui = str(r.cui)
    gold_cui = cui if cui.startswith("UMLS:") else f"UMLS:{cui}"
    rows.append(
        {
            "instance_id": f"cadec_{r.doc_id}_{r.annotation_id}",
            "document_id": r.doc_id,
            "full_original_text": full,
            "mention_context": ctx,
            "gold_mention": gold_mention,
            "gold_cui": gold_cui,
            "semantic_type": r.semantic_type if pd.notna(r.semantic_type) else "",
            "mention_start": int(r.char_start),
            "mention_end": int(r.char_end),
            "original_text": ctx,
        }
    )

df_inst = pd.DataFrame(rows)[OUT_COLS]
print(f"Writing: {OUT_CSV}")
df_inst.to_csv(OUT_CSV, index=False)
_log(f"Wrote {len(df_inst):,} instances → {OUT_CSV}")
print("Output columns:", list(df_inst.columns))
assert list(df_inst.columns) == OUT_COLS
assert all(c in df_inst.columns for c in CORE_COLS)
assert "question" not in df_inst.columns and "gold_answer" not in df_inst.columns
print(df_inst[CORE_COLS + [c for c in ["document_id", "semantic_type"] if c in df_inst.columns]].head(3).to_string())
sys.stdout.flush()

[2026-07-28 02:20:06 UTC] Loaded texts for 1,181 docs (missing .txt=0)


Writing: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_instances.csv


[2026-07-28 02:20:06 UTC] Wrote 7,006 instances → /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_instances.csv


Output columns: ['instance_id', 'document_id', 'full_original_text', 'mention_context', 'gold_mention', 'gold_cui', 'semantic_type', 'mention_start', 'mention_end', 'original_text']
             instance_id                                                                  mention_context           gold_mention       gold_cui  document_id semantic_type
0  cadec_ARTHROTEC.1_TT1  I feel a bit drowsy & have a little blurred vision, so far no gastric problems.             bit drowsy  UMLS:C0013144  ARTHROTEC.1          T033
1  cadec_ARTHROTEC.1_TT2  I feel a bit drowsy & have a little blurred vision, so far no gastric problems.  little blurred vision  UMLS:C0344232  ARTHROTEC.1          T033
2  cadec_ARTHROTEC.1_TT4  I feel a bit drowsy & have a little blurred vision, so far no gastric problems.       gastric problems  UMLS:C0016204  ARTHROTEC.1          T184


## 4) Final counts and assertions

In [5]:
# === Summary ================================================================
n_docs = int(df_mentions["doc_id"].nunique())
n_lines_total = int(sum(v for k, v in drop_counts.items() if k != "empty"))
n_with_cui = len(df_inst)
n_unique_cui = int(df_inst["gold_cui"].nunique())
n_unique_docs_out = int(df_inst["document_id"].nunique())
n_other_drops = (
    n_dropped
    - drop_counts.get("concept_less", 0)
    - drop_counts.get("no_valid_sctid", 0)
)

print("=" * 60)
print("CADEC adapter summary")
print("=" * 60)
print(f"Annotation files (docs with .ann): {len(ann_files):,}")
print(f"Docs represented in kept mentions:  {n_docs:,}")
print(f"Docs in output instances:           {n_unique_docs_out:,}")
print(f"Ann lines (excl. empty):            {n_lines_total:,}")
print(f"Mentions kept (valid SCTID):        {n_kept:,}")
print(f"Mentions dropped (parse):           {n_dropped:,}")
print(f"  of which CONCEPT_LESS:            {drop_counts.get('concept_less', 0):,}")
print(f"  of which no_valid_sctid:          {drop_counts.get('no_valid_sctid', 0):,}")
print(f"  of which other parse drops:       {n_other_drops:,}")
print(f"CUI map OK / fail:                  {n_map_ok:,} / {n_map_fail:,}")
print(f"Unique unmapped SCTIDs:             {n_unique_sct_fail:,}")
print(f"Instances with valid CUI (output):  {n_with_cui:,}")
print(f"Unique CUIs in output:              {n_unique_cui:,}")
print(f"Output: {OUT_CSV}")
print("=" * 60)
sys.stdout.flush()

assert n_with_cui > 0, "ASSERT FAIL: zero CADEC instances with valid CUIs"
assert OUT_CSV.is_file() and OUT_CSV.stat().st_size > 0
_log("ASSERT OK: CADEC instances written with >0 valid CUIs.")

CADEC adapter summary
Annotation files (docs with .ann): 1,250
Docs represented in kept mentions:  1,181
Docs in output instances:           1,152
Ann lines (excl. empty):            9,111
Mentions kept (valid SCTID):        8,665
Mentions dropped (parse):           446
  of which CONCEPT_LESS:            445
  of which no_valid_sctid:          1
  of which other parse drops:       0
CUI map OK / fail:                  7,006 / 1,659
Unique unmapped SCTIDs:             125
Instances with valid CUI (output):  7,006
Unique CUIs in output:              881
Output: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_cadec_instances.csv


[2026-07-28 02:20:06 UTC] ASSERT OK: CADEC instances written with >0 valid CUIs.
